In [1]:
!apt-get install openjdk-17-jdk-headless -qq > /dev/null
!pip install pyspark spacy -qq
!python -m spacy download en_core_web_sm -qq

import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 122.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
!pip install gdown -q
import gdown, os

CATEGORY_NAME = 'Kindle_Store'

PHASE1_FILE_ID = '1wKUBzQE0W9fnrVfD19DLACSNTQrZxmba' 
gdown.download(id=PHASE1_FILE_ID, output='/content/output_data.zip', quiet=False)
!unzip -qo /content/output_data.zip -d /content/


Downloading...
From (original): https://drive.google.com/uc?id=1wKUBzQE0W9fnrVfD19DLACSNTQrZxmba
From (redirected): https://drive.google.com/uc?id=1wKUBzQE0W9fnrVfD19DLACSNTQrZxmba&confirm=t&uuid=f3c57444-c5d3-4716-856f-a185dac4dae1
To: /content/output_data.zip
100%|██████████| 4.50G/4.50G [00:51<00:00, 87.2MB/s]


In [21]:
INPUT_PARQUET = f'/content/content/outputs/cleaned_{CATEGORY_NAME}.parquet'
OUTPUT_DIR = '/content/outputs'
OUTPUT_PARQUET = f'{OUTPUT_DIR}/cleaned2_{CATEGORY_NAME}.parquet'
REPORT_PATH = f'{OUTPUT_DIR}/reports/reviews_cleaning2_report_{CATEGORY_NAME}.txt'

In [22]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StringType, BooleanType, StructType, StructField

spark = SparkSession.builder \
    .appName('Kindle Review Cleaning Phase 2') \
    .master('local[*]') \
    .config('spark.driver.memory', '40g') \
    .config('spark.driver.maxResultSize', '10g') \
    .config('spark.sql.shuffle.partitions', '400') \
    .config('spark.sql.adaptive.enabled', 'true') \
    .config('spark.sql.adaptive.coalescePartitions.enabled', 'true') \
    .config('spark.serializer', 'org.apache.spark.serializer.KryoSerializer') \
    .config('spark.sql.parquet.compression.codec', 'gzip') \
    .config('spark.default.parallelism', '400') \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN')

df = spark.read.parquet(INPUT_PARQUET)
initial_sentence_count = df.count()
initial_review_count = df.select('review_id').distinct().count()
df.show(5, truncate=80)

+-----------+---------+-----------+--------------------------------------------------------------------------------+------+
|parent_asin|review_id|sentence_id|                                                                   sentence_text|rating|
+-----------+---------+-----------+--------------------------------------------------------------------------------+------+
| B00TDNMBPC|      216|          1|                                        fun and funny for parents and children!.|   5.0|
| B00TDNMBPC|      216|          2|                                                      kids enjoyed it very much.|   5.0|
| B00TDNMBPC|      216|          3|its got a very interesting and unique narrative/dialogue structure that lets ...|   5.0|
| B00TDNMBPC|      216|          4|                                          fun and funny for parents and children|   5.0|
| B0127HOQI0|      393|          1|                                                               What and where ?.|   3.0|
+-------

In [23]:
CONTRACTIONS = {
    "won't": "will not", "can't": "can not", "shan't": "shall not",
    "n't": " not", "'re": " are", "'ve": " have",
    "'ll": " will", "'d": " would", "'m": " am",
    "let's": "let us", "it's": "it is", "that's": "that is",
    "what's": "what is", "there's": "there is", "here's": "here is",
    "who's": "who is", "how's": "how is", "where's": "where is",
    "he's": "he is", "she's": "she is",
    "i'm": "i am", "you're": "you are", "we're": "we are", "they're": "they are",
    "i've": "i have", "you've": "you have", "we've": "we have", "they've": "they have",
    "i'll": "i will", "you'll": "you will", "he'll": "he will", "she'll": "she will",
    "we'll": "we will", "they'll": "they will",
    "i'd": "i would", "you'd": "you would", "he'd": "he would", "she'd": "she would",
    "we'd": "we would", "they'd": "they would",
    "isn't": "is not", "aren't": "are not", "wasn't": "was not", "weren't": "were not",
    "hasn't": "has not", "haven't": "have not", "hadn't": "had not",
    "doesn't": "does not", "don't": "do not", "didn't": "did not",
    "wouldn't": "would not", "shouldn't": "should not", "couldn't": "could not",
    "mustn't": "must not", "needn't": "need not",
}

bc_contractions = spark.sparkContext.broadcast(CONTRACTIONS)

@F.udf(StringType())
def normalize_text(text):
    if not text:
        return text
    import re
    text = text.lower()
    contractions = bc_contractions.value
    for k, v in sorted(contractions.items(), key=lambda x: -len(x[0])):
        text = text.replace(k, v)

    text = re.sub(r"[^a-z0-9\s']", ' ', text)
    text = re.sub(r"\s+", ' ', text).strip()
    return text

df = df.withColumn('sentence_text', normalize_text('sentence_text'))
df = df.filter(
    (F.col('sentence_text').isNotNull()) &
    (F.trim('sentence_text') != '')
)
after_normalize = df.cache().count()
print('Chuẩn hóa text. Còn {after_normalize:,} câu')
df.show(5, truncate=80)

Chuẩn hóa text. Còn {after_normalize:,} câu
+-----------+---------+-----------+--------------------------------------------------------------------------------+------+
|parent_asin|review_id|sentence_id|                                                                   sentence_text|rating|
+-----------+---------+-----------+--------------------------------------------------------------------------------+------+
| B00TDNMBPC|      216|          1|                                          fun and funny for parents and children|   5.0|
| B00TDNMBPC|      216|          2|                                                       kids enjoyed it very much|   5.0|
| B00TDNMBPC|      216|          3|its got a very interesting and unique narrative dialogue structure that lets ...|   5.0|
| B00TDNMBPC|      216|          4|                                          fun and funny for parents and children|   5.0|
| B0127HOQI0|      393|          1|                                                     

In [24]:
GENERIC_NOUNS = {
    'thing', 'things', 'stuff', 'item', 'items', 'product', 'products',
    'object', 'objects', 'part', 'parts', 'piece', 'pieces',
    'kind', 'kinds', 'type', 'types', 'sort', 'sorts',
    'one', 'ones', 'something', 'anything', 'everything', 'nothing',
    'someone', 'anyone', 'everyone', 'nobody',
    'aspect', 'issue', 'issues', 'lot', 'bit', 'matter', 'way',
    'case', 'point', 'side', 'version',
    'time', 'times', 'day', 'days', 'week', 'weeks',
    'month', 'months', 'year', 'years',
    'problem', 'problems', 'reason', 'reasons', 'result', 'results',
    'experience', 'review', 'reviews', 'rating', 'ratings', 'star', 'stars'
}

DOMAIN_NOISE = {
    'amazon', 'seller', 'sellers', 'shipping', 'delivery',
    'shipment', 'shipments', 'package', 'packages', 'packaging',
    'box', 'boxes', 'wrapper', 'wrappers',
    'return', 'returns', 'refund', 'refunds',
    'replacement', 'replacements'
}

bc_generic = spark.sparkContext.broadcast(GENERIC_NOUNS)
bc_noise = spark.sparkContext.broadcast(DOMAIN_NOISE)

@F.udf(StringType())
def tag_special_words(text):
    if not text:
        return text
    generic = bc_generic.value
    noise = bc_noise.value
    words = text.split()
    result = []
    for w in words:
        w_clean = w.strip("'")
        if w_clean in generic:
            result.append('[GENERIC_NOUN]')
        elif w_clean in noise:
            result.append('[DOMAIN_NOISE]')
        else:
            result.append(w)
    return ' '.join(result)

df.unpersist()
df = df.withColumn('sentence_text', tag_special_words('sentence_text'))
print('Đánh dấu GENERIC_NOUN và DOMAIN_NOISE')
df.show(5, truncate=80)

Đánh dấu GENERIC_NOUN và DOMAIN_NOISE
+-----------+---------+-----------+--------------------------------------------------------------------------------+------+
|parent_asin|review_id|sentence_id|                                                                   sentence_text|rating|
+-----------+---------+-----------+--------------------------------------------------------------------------------+------+
| B00TDNMBPC|      216|          1|                                          fun and funny for parents and children|   5.0|
| B00TDNMBPC|      216|          2|                                                       kids enjoyed it very much|   5.0|
| B00TDNMBPC|      216|          3|its got a very interesting and unique narrative dialogue structure that lets ...|   5.0|
| B00TDNMBPC|      216|          4|                                          fun and funny for parents and children|   5.0|
| B0127HOQI0|      393|          1|                                                           

In [25]:
generic_count = df.select(
    F.sum(F.size(F.split(F.col('sentence_text'), '\\[GENERIC_NOUN\\]')) - 1)
).collect()[0][0] or 0

noise_count = df.select(
    F.sum(F.size(F.split(F.col('sentence_text'), '\\[DOMAIN_NOISE\\]')) - 1)
).collect()[0][0] or 0

print(f'Số lượng GENERIC_NOUN tags: {generic_count:,}')
print(f'Số lượng DOMAIN_NOISE tags: {noise_count:,}')

Số lượng GENERIC_NOUN tags: 44,399,217
Số lượng DOMAIN_NOISE tags: 630,804


In [26]:
df_short = df.filter(
    (F.length('sentence_text') < 5) |
    (F.size(F.split(F.col('sentence_text'), '\\s+')) < 2)
)
short_count = df_short.count()

df = df.filter(
    (F.length('sentence_text') >= 5) &
    (F.size(F.split(F.col('sentence_text'), '\\s+')) >= 2)
)

after_short = df.cache().count()
print(f'Xóa {short_count:,} câu quá ngắn. Còn {after_short:,} câu')

Xóa 4,802,700 câu quá ngắn. Còn 140,718,260 câu


In [27]:
PRONOUNS = {'i', 'me', 'my', 'mine', 'myself',
            'you', 'your', 'yours', 'yourself',
            'he', 'him', 'his', 'himself',
            'she', 'her', 'hers', 'herself',
            'it', 'its', 'itself',
            'we', 'us', 'our', 'ours', 'ourselves',
            'they', 'them', 'their', 'theirs', 'themselves',
            'this', 'that', 'these', 'those'}

OPINION_TAGS = {'JJ', 'JJR', 'JJS'}
NOUN_TAGS = {'NN', 'NNS', 'NNP', 'NNPS'}
LINKING_VERBS = {'be', 'is', 'am', 'are', 'was', 'were', 'been', 'being',
                 'feel', 'seem', 'appear', 'look', 'sound', 'taste', 'smell',
                 'become', 'get', 'grow', 'turn', 'remain', 'stay'}

bc_pronouns = spark.sparkContext.broadcast(PRONOUNS)
bc_opinion_tags = spark.sparkContext.broadcast(OPINION_TAGS)
bc_noun_tags = spark.sparkContext.broadcast(NOUN_TAGS)
bc_linking = spark.sparkContext.broadcast(LINKING_VERBS)

In [28]:
from pyspark.sql import Row

def process_partition(rows):
    import spacy
    nlp = spacy.load('en_core_web_sm', disable=['ner', 'lemmatizer'])
    pronouns = bc_pronouns.value
    opinion_tags = bc_opinion_tags.value
    noun_tags = bc_noun_tags.value
    linking = bc_linking.value

    def has_noun(doc):
        return any(t.tag_ in noun_tags for t in doc)

    def only_pronoun_target(doc):
        subjects = [t for t in doc if t.dep_ in ('nsubj', 'nsubjpass')]
        if not subjects:
            return False
        return all(t.text.lower() in pronouns for t in subjects)

    def all_targets_generic_noise(doc):
        targets = [t for t in doc if t.dep_ in ('nsubj', 'nsubjpass', 'dobj', 'obj')]
        if not targets:
            return True
        return all(t.text in ('[GENERIC_NOUN]', '[DOMAIN_NOISE]') for t in targets)

    def matches_dep_pattern(doc):
        for token in doc:
            if token.dep_ == 'amod' and token.tag_ in opinion_tags:
                if token.head.tag_ in noun_tags:
                    return True

            if token.dep_ == 'nsubj' and token.tag_ in noun_tags:
                if token.head.tag_ in opinion_tags or token.head.tag_.startswith('VB'):
                    return True

            if token.dep_ in ('dobj', 'obj') and token.tag_ in noun_tags:
                if token.head.tag_.startswith('VB'):
                    return True

            if token.tag_ in opinion_tags and token.dep_ in ('acomp', 'attr'):
                if token.head.lemma_ in linking:
                    subj = [c for c in token.head.children if c.dep_ == 'nsubj']
                    if subj:
                        return True

            if token.dep_ == 'conj':
                if token.tag_ in opinion_tags and token.head.tag_ in opinion_tags:
                    return True

                if token.tag_ in noun_tags and token.head.tag_ in noun_tags:
                    return True

            if token.dep_ == 'neg':
                if token.head.tag_ in opinion_tags or token.head.tag_.startswith('VB'):
                    return True

            if token.dep_ == 'xcomp' and token.tag_ in opinion_tags:
                if token.head.tag_.startswith('VB'):
                    obj_children = [c for c in token.head.children if c.dep_ in ('dobj', 'obj')]
                    if obj_children:
                        return True
        return False

    batch_texts = []
    batch_rows = []
    BATCH_SIZE = 2048

    def flush():
        if not batch_texts:
            return
        docs = list(nlp.pipe(batch_texts, batch_size=512))
        for row, doc in zip(batch_rows, docs):
            if not has_noun(doc):
                label = 'no_noun'
            elif only_pronoun_target(doc):
                label = 'pronoun_only'
            elif all_targets_generic_noise(doc):
                label = 'generic_noise_only'
            elif not matches_dep_pattern(doc):
                label = 'no_pattern'
            else:
                label = 'keep'
            yield Row(
                parent_asin=row['parent_asin'],
                review_id=int(row['review_id']),
                sentence_id=int(row['sentence_id']),
                sentence_text=row['sentence_text'],
                rating=float(row['rating']),
                filter_label=label
            )

    for row in rows:
        text = row['sentence_text']
        parse_text = text.replace('[GENERIC_NOUN]', 'thing').replace('[DOMAIN_NOISE]', 'shipping')
        batch_texts.append(parse_text)
        batch_rows.append(row)
        if len(batch_texts) >= BATCH_SIZE:
            yield from flush()
            batch_texts = []
            batch_rows = []
    yield from flush()


In [ ]:
num_cores = os.cpu_count()
df = df.repartition(num_cores * 2)

df_labeled = df.rdd.mapPartitions(process_partition).toDF()
df_labeled.cache()

stats = df_labeled.groupBy('filter_label').count().collect()
stats_dict = {row['filter_label']: row['count'] for row in stats}

no_noun_count = stats_dict.get('no_noun', 0)
pronoun_only_count = stats_dict.get('pronoun_only', 0)
generic_noise_count = stats_dict.get('generic_noise_only', 0)
no_pattern_count = stats_dict.get('no_pattern', 0)
keep_count = stats_dict.get('keep', 0)

print(f'  Kết quả lọc spacy:')
print(f'  Không có danh từ:           {no_noun_count:,}')
print(f'  Chỉ pronoun làm target:     {pronoun_only_count:,}')
print(f'  Toàn generic/noise target:  {generic_noise_count:,}')
print(f'  Không khớp dep pattern:     {no_pattern_count:,}')
print(f'  Giữ lại:                    {keep_count:,}')

Repartition thành 16 partitions (8 CPU cores)
  Kết quả lọc spacy:
  Không có danh từ:           12,956,324
  Chỉ pronoun làm target:     40,417,828
  Toàn generic/noise target:  24,327,048
  Không khớp dep pattern:     2,059,263
  Giữ lại:                    60,957,797


In [30]:
from pyspark.sql.window import Window

df_clean = df_labeled.filter(F.col('filter_label') == 'keep').drop('filter_label')

w = Window.partitionBy('review_id').orderBy('sentence_id')
df_clean = df_clean.withColumn('sentence_id', F.row_number().over(w))

df_final = df_clean.select('parent_asin', 'review_id', 'sentence_id', 'sentence_text', 'rating')
df_final.cache()

final_sentence_count = df_final.count()
final_review_count = df_final.select('review_id').distinct().count()
print(f'Còn {final_sentence_count:,} câu từ {final_review_count:,} review')
df_final.show(10, truncate=80)

Còn 60,957,797 câu từ 19,858,252 review
+-----------+---------+-----------+--------------------------------------------------------------------------------+------+
|parent_asin|review_id|sentence_id|                                                                   sentence_text|rating|
+-----------+---------+-----------+--------------------------------------------------------------------------------+------+
| B005KGJXAW|     2040|          1|dragonbreath travels to the sargasso sea in order to write a good paper for s...|   2.0|
| B005KGJXAW|     2040|          2|                                               not sure how kids feel about this|   2.0|
| B08B4ZPQ9W|     2250|          1|as two opposites attract who are so full of life and love fire and passion an...|   5.0|
| B08B4ZPQ9W|     2250|          2|youll fall in love with mercy goode a young woman who defies societys limits ...|   5.0|
| B08B4ZPQ9W|     2250|          3|and for the first [GENERIC_NOUN] in her life he made merc

In [31]:
os.makedirs(os.path.dirname(OUTPUT_PARQUET), exist_ok=True)
df_final.write.mode('overwrite').parquet(OUTPUT_PARQUET)
print(f'Đã lưu parquet tại: {OUTPUT_PARQUET}')

Đã lưu parquet tại: /content/outputs/cleaned2_Kindle_Store.parquet


In [32]:
def get_dir_size(path):
    total = 0
    for dirpath, dirnames, filenames in os.walk(path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total += os.path.getsize(fp)
    return total

def format_size(size_bytes):
    if size_bytes >= 1024**3:
        return f'{size_bytes / 1024**3:.2f} GB'
    elif size_bytes >= 1024**2:
        return f'{size_bytes / 1024**2:.2f} MB'
    return f'{size_bytes / 1024:.2f} KB'

output_size = get_dir_size(OUTPUT_PARQUET)
useful_ratio = final_review_count / initial_review_count * 100

report = f"""Báo cáo: {CATEGORY_NAME}
  Số câu ban đầu:        {initial_sentence_count:,}
  Số review ban đầu:     {initial_review_count:,}
  Sau chuẩn hóa:         {after_normalize:,}
  Số lượng GENERIC_NOUN: {generic_count:,}
  Số lượng DOMAIN_NOISE: {noise_count:,}
  Câu quá ngắn (< 5 ký tự hoặc < 2 token):   {short_count:,}
  Câu không có danh từ:                      {no_noun_count:,}
  Câu chỉ có pronoun làm target:             {pronoun_only_count:,}
  Câu toàn generic/noise target:             {generic_noise_count:,}
  Câu không khớp dependency pattern:         {no_pattern_count:,}
  Số câu sau xử lý:            {final_sentence_count:,}
  Số review còn lại:           {final_review_count:,}
  Kích thước output:           {format_size(output_size)}
"""

os.makedirs(os.path.dirname(REPORT_PATH), exist_ok=True)
with open(REPORT_PATH, 'w', encoding='utf-8') as f:
    f.write(report)

print(report)

Báo cáo: Kindle_Store
  Số câu ban đầu:        146,902,993
  Số review ban đầu:     25,262,592
  Sau chuẩn hóa:         145,520,960
  Số lượng GENERIC_NOUN: 44,399,217
  Số lượng DOMAIN_NOISE: 630,804
  Câu quá ngắn (< 5 ký tự hoặc < 2 token):   4,802,700
  Câu không có danh từ:                      12,956,324
  Câu chỉ có pronoun làm target:             40,417,828
  Câu toàn generic/noise target:             24,327,048
  Câu không khớp dependency pattern:         2,059,263
  Số câu sau xử lý:            60,957,797
  Số review còn lại:           19,858,252
  Kích thước output:           2.34 GB



In [33]:
!zip -r /content/output_data_phase2.zip /content/outputs

from google.colab import files
files.download('/content/output_data_phase2.zip')

  adding: content/outputs/ (stored 0%)
  adding: content/outputs/cleaned2_Kindle_Store.parquet/ (stored 0%)
  adding: content/outputs/cleaned2_Kindle_Store.parquet/.part-00316-cbf1435b-2112-4fc9-8f50-3324cd857931-c000.gz.parquet.crc (deflated 0%)
  adding: content/outputs/cleaned2_Kindle_Store.parquet/part-00238-cbf1435b-2112-4fc9-8f50-3324cd857931-c000.gz.parquet (deflated 0%)
  adding: content/outputs/cleaned2_Kindle_Store.parquet/.part-00277-cbf1435b-2112-4fc9-8f50-3324cd857931-c000.gz.parquet.crc (deflated 0%)
  adding: content/outputs/cleaned2_Kindle_Store.parquet/.part-00148-cbf1435b-2112-4fc9-8f50-3324cd857931-c000.gz.parquet.crc (deflated 0%)
  adding: content/outputs/cleaned2_Kindle_Store.parquet/.part-00358-cbf1435b-2112-4fc9-8f50-3324cd857931-c000.gz.parquet.crc (deflated 0%)
  adding: content/outputs/cleaned2_Kindle_Store.parquet/part-00020-cbf1435b-2112-4fc9-8f50-3324cd857931-c000.gz.parquet (deflated 0%)
  adding: content/outputs/cleaned2_Kindle_Store.parquet/part-00002-c

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>